# Kaggle Runner — SHAP ECG Arrhythmia (TCC)

Runner fino: clona/atualiza o repositório e chama os scripts de `src/`.
Toda a lógica vive no repositório; este notebook apenas orquestra a execução
na GPU do Kaggle.

O dataset **CPSC2018** já vem anexado via `dataset_sources` no
`kernel-metadata.json` (`physionet/china-physiological-signal-challenge-in-2018`),
montado em `/kaggle/input/...`. Basta ativar **GPU** e **Internet** nas
configurações do notebook (o clone do GitHub exige internet).

In [ ]:
# Célula 1 — clonar / atualizar o repositório
import os

REPO_URL = 'https://github.com/Arthur-so/shap-ecg-arrhythmia.git'
REPO_DIR = '/kaggle/working/shap-ecg-arrhythmia'

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
# Célula 2 — instalar dependências (torch já vem no Kaggle)
!pip install -q -r requirements.txt

In [ ]:
# Célula 3 — configuração de caminhos
# O CPSC2018 vem anexado como Kaggle Dataset (ver dataset_sources no
# kernel-metadata.json). Ele é montado em /kaggle/input/<slug>/, com os
# registros WFDB em Training_WFDB/ e os rótulos SNOMED no campo #Dx: de cada
# .hea (o preprocess.py detecta esse formato automaticamente).
RAW_DIR = '/kaggle/input/china-physiological-signal-challenge-in-2018'
PROC_DIR = '/kaggle/working/processed'
CKPT_DIR = '/kaggle/working/checkpoints'
ATTR_DIR = '/kaggle/working/results/attributions'
RESULTS_DIR = 'results'

import torch, os
print('CUDA disponível:', torch.cuda.is_available())
print('RAW_DIR existe:', os.path.isdir(RAW_DIR))

In [ ]:
# Célula 4 — pré-processamento (truncamento/padding, z-score, split)
!python -m src.data.preprocess --raw-dir {RAW_DIR} --out-dir {PROC_DIR}

In [ ]:
# Célula 5 — treino da ResNet34 1D
!python -m src.train \
    --data-dir {PROC_DIR} \
    --out-dir {CKPT_DIR} \
    --results-dir {RESULTS_DIR} \
    --epochs 50 --lr 1e-3 --batch-size 32 --patience 10

In [ ]:
# Célula 6 — geração das explicações GradientSHAP
!python -m src.explain \
    --data-dir {PROC_DIR} \
    --checkpoint {CKPT_DIR}/resnet34_1d_best.pt \
    --out-dir {ATTR_DIR}

In [ ]:
# Célula 7 — avaliação Quantus (Faithfulness / Robustness por classe)
!python -m src.evaluate \
    --data-dir {PROC_DIR} \
    --checkpoint {CKPT_DIR}/resnet34_1d_best.pt \
    --attr-dir {ATTR_DIR} \
    --results-dir {RESULTS_DIR} \
    --max-per-class 50

In [ ]:
# Célula 8 — inspeção rápida dos resultados agregados
import pandas as pd
print(pd.read_csv(f'{RESULTS_DIR}/quality_vs_f1.csv').to_string(index=False))